In [2]:
import pickle
import numpy as np
import networkx as nx
from src.plotting import *
from src.utils.graph_utils import low_energy_edge_stats

from scipy.stats import t, ttest_rel

def welchs_t_test_onesided(mean1, std1, n1, mean2, std2, n2):
    # H_0: mean1 <= mean2
    # H_1: mean1 > mean2
    # Step 1: Compute test statistic
    se1 = std1**2 / n1
    se2 = std2**2 / n2
    t_stat = (mean1 - mean2) / np.sqrt(se1 + se2)
    
    # Step 2: Degrees of freedom (Welch–Satterthwaite)
    numerator = (se1 + se2)**2
    denominator = (se1**2 / (n1 - 1)) + (se2**2 / (n2 - 1))
    df = numerator / denominator

    # Step 3: One-sided p-value
    p_value = 1 - t.cdf(t_stat, df)

    return t_stat, df, p_value

def extract_graphs(data_dict):
    nodes = np.array(data_dict["embedor_node_info"])
    edges = np.array(data_dict["embedor_edge_info"])
    embedor_emb = np.array(data_dict["embedor_emb"])
    embedor_apsp = np.array(data_dict["embedor_apsp"])
    # create networkx graph
    G = nx.Graph()
    G_low_energy = nx.Graph()

    for i, node in enumerate(nodes):
        G.add_node(i, emb=embedor_emb[i])
        G_low_energy.add_node(i, emb=embedor_emb[i])
    # add edges
    embedor_distances = []
    for edge in edges:
        G.add_edge(edge[0], edge[1])
        embedor_distances.append(embedor_apsp[edge[0], edge[1]])
    embedor_distances = np.array(embedor_distances)
    sorted_indices = np.argsort(embedor_distances)
    bottom_indices = sorted_indices[:int(len(sorted_indices)//3)]
    bottom_edges = [pair for i, pair in enumerate(edges) if i in bottom_indices]
    G_low_energy.add_edges_from(bottom_edges)
    return G, G_low_energy

In [3]:
## paired t-test

datasets = ['mnist', 'fmnist', 'developmental', 'macosko', 'chimp']

for dataset in datasets:
    print("*"*100)
    print(f"Processing dataset: {dataset}")
    dict_path = f"/home/tristan/Research/Sp25/embedor/outputs/server_experiments/embedor_{dataset}_25k.pkl"
    with open(dict_path, "rb") as f:
        data = pickle.load(f)

    G, G_low_energy = extract_graphs(data)
    _, _, _, z_scores = low_energy_edge_stats(data['embedor_emb'], G, G_low_energy)
    _, _, _, z_scores_umap = low_energy_edge_stats(data['umap_emb'], G, G_low_energy)
    _, _, _, z_scores_tsne = low_energy_edge_stats(data['tsne_emb'], G, G_low_energy)

    t_stat, p_val = ttest_rel(z_scores, z_scores_umap)

    print(f"Paired t-test (H₀: μ_emb ≥ μ_umap, H₁: μ_emb < μ_umap): "
      f"t = {t_stat:.4f}, p = {p_val:.4g} — "
      f"{'REJECT' if p_val < 0.05 else 'FAIL TO REJECT'} H₀ at α = 0.05")
    
    t_stat, p_val = ttest_rel(z_scores, z_scores_tsne)
    print(f"Paired t-test (H₀: μ_emb ≥ μ_tsne, H₁: μ_emb < μ_tsne): "
      f"t = {t_stat:.4f}, p = {p_val:.4g} — "
      f"{'REJECT' if p_val < 0.05 else 'FAIL TO REJECT'} H₀ at α = 0.05")
    print("*"*100)
    print()
    

****************************************************************************************************
Processing dataset: mnist
Paired t-test (H₀: μ_emb ≥ μ_umap, H₁: μ_emb < μ_umap): t = -152.6604, p = 0 — reject H₀ at α = 0.05
Paired t-test (H₀: μ_emb ≥ μ_tsne, H₁: μ_emb < μ_tsne): t = 2.2842, p = 0.02236 — reject H₀ at α = 0.05
****************************************************************************************************

****************************************************************************************************
Processing dataset: fmnist
Paired t-test (H₀: μ_emb ≥ μ_umap, H₁: μ_emb < μ_umap): t = -218.0809, p = 0 — reject H₀ at α = 0.05
Paired t-test (H₀: μ_emb ≥ μ_tsne, H₁: μ_emb < μ_tsne): t = -29.7718, p = 6.454e-194 — reject H₀ at α = 0.05
****************************************************************************************************

****************************************************************************************************
Processing dataset: devel